# Smoketest for the test-split performance benchmark (few minutes, Kaggle T4)

Validates exactly what `testing/index.kaggle.ipynb` needs: T4 GPU + torch,
the HF token (for the card update), BOTH staged test tars mounted +
extraction, model download, and a mini run through the same batched-fp16
measurement pipeline (2 sequences per part, large stride). Mounts are
identical to `index.config.json` on purpose.

In [ ]:
# --- 1. Dependencies --------------------------------------------------------
# CRITICAL: do NOT reinstall or upgrade torch OR transformers (torch: GPU
# kernel compatibility; transformers: the checkpoint must load under the
# lineage that trained it). Nothing extra is needed here — performance
# benchmarking uses only the preinstalled stack.
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
# --- 2. Hugging Face token --------------------------------------------------
# Needed WITH WRITE ACCESS to publish the benchmark section to the model
# card; without it the run still completes and benchmarks.json stays in the
# kernel output. Rides in the private external-secrets dataset (Kaggle
# Secrets are dropped on every push; dataset mounts persist). The mount
# layout has changed before, so scan /kaggle/input instead of hard-coding.
import os
from pathlib import Path

def _read_hf_token():
    base = Path("/kaggle/input")
    hits = sorted(base.rglob("secrets")) if base.is_dir() else []
    for hit in hits:
        for line in hit.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    return None

_tok = _read_hf_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
print("HF auth:", "token found" if _tok else "none (card will NOT be updated)")

In [ ]:
# --- 3. GPU sanity ----------------------------------------------------------
# Fail fast on the wrong accelerator: the API-default P100 (sm_60) has no CUDA
# kernels in Kaggle's torch build; configs pin machine_shape=NvidiaTeslaT4.
assert torch.cuda.is_available(), "No CUDA GPU — check the kernel's accelerator settings"
cap = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"GPU: {name} (sm_{cap[0]}{cap[1]})")
assert cap >= (7, 0), f"{name} unusable: Kaggle torch ships no kernels for it"
x = torch.randn(256, 256, device="cuda") @ torch.randn(256, 256, device="cuda")
torch.cuda.synchronize()
print("CUDA matmul OK:", float(x.sum()))

In [ ]:
# --- 4. Locate BOTH staged test parts ----------------------------------------
# The full test split is staged as two mutually exclusive halves (Kaggle's
# 20 GB kernel-output cap): sportsmot-test-1.tar (even-indexed sequences) and
# sportsmot-test-2.tar (odd-indexed), mounted via kernel_sources. Both are
# REQUIRED — benchmarking a silent subset would misreport coverage. Each part
# extracts into ITS OWN directory (a shared dir + background extraction once
# deleted part 1 mid-benchmark) and is removed after processing.
import shutil
import subprocess
import time

def _find_tar(part):
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for pattern in (f"*/sportsmot-test-{part}.tar", f"*/*/sportsmot-test-{part}.tar",
                    f"*/*/*/sportsmot-test-{part}.tar"):
        hits = sorted(base.glob(pattern))
        if hits:
            return hits[0]
    return None

TARS = {part: _find_tar(part) for part in (1, 2)}
for part, tar in TARS.items():
    assert tar is not None, (f"sportsmot-test-{part}.tar not mounted — run "
                             f"testing/prepare-data-{part} and keep both staging "
                             "slugs in this config's kernel_sources")
    print(f"part {part}: {tar}")

def part_dir(part):
    return Path(f"/tmp/sportsmot-part{part}")

def extract_part(part):
    """Extract one staged half into its own directory; returns its seq dirs."""
    dest = part_dir(part)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    _t = time.time()
    subprocess.run(["tar", "-xf", str(TARS[part]), "-C", str(dest)], check=True)
    seqs = sorted(p for p in (dest / "test").iterdir() if (p / "img1").is_dir())
    print(f"part {part}: extracted {len(seqs)} sequences in {time.time() - _t:.0f}s",
          flush=True)
    return seqs

In [ ]:
# --- 5. Load the fine-tuned model from the Hub ------------------------------
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_ID = "smallTech/rtdetr-sportsmot"
_t = time.time()
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID).to("cuda").eval()
LOAD_S = time.time() - _t
print(f"loaded {MODEL_ID} in {LOAD_S:.1f}s "
      f"({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params)")

In [ ]:
# --- 6. Performance benchmark over BOTH parts --------------------------------
# MAX T4 UTILIZATION: a single 640x640 image nowhere near saturates the GPU's
# SMs, so the pipeline batches 8 images per forward pass under fp16
# autocast (Turing tensor cores) — that is how "all the cores" get used.
# Image decode/preprocess overlaps GPU compute via a thread pool (Kaggle GPU
# kernels have 4 vCPUs). Two distinct numbers are measured:
#   * THROUGHPUT — frames/s of the batched, fp16, prefetched pipeline: the
#     utilization-maximizing number;
#   * single-image LATENCY percentiles — a separate batch-1 probe over the
#     first 10 frames, since latency and throughput answer different
#     questions and batching trades the former for the latter.
# The test split has NO ground truth, so accuracy metrics are impossible —
# detections/frame and confidence stats are tracked instead.
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

EVAL_STRIDE = 25
CONF = 0.5
BATCH = 8
PROBE_N = 10

def load_rgb(path):
    return Image.open(path).convert("RGB")

@torch.no_grad()
def run_batch(images):
    """One batched forward: preprocess -> fp16 forward -> postprocess."""
    sizes = torch.tensor([im.size[::-1] for im in images]).to("cuda")
    inputs = processor(images=images, return_tensors="pt").to("cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(**inputs)
    return processor.post_process_object_detection(
        outputs, target_sizes=sizes, threshold=CONF)

# single-image latency probe (batch 1, same fp16 path), then the batched run
latencies_1 = []
probe_done = False
det_counts, score_sums = [], []
per_seq = {}
n_frames = 0
gpu_s = 0.0
t_bench = time.time()
# Overlap I/O with compute: extract part 2 in the background while the GPU
# processes part 1 (both halves fit on disk simultaneously; each is deleted
# right after processing).
import threading
_bg = {}
_t2 = threading.Thread(target=lambda: _bg.update({2: extract_part(2)}))
_parts_seqs = {1: extract_part(1)}
_t2.start()
for _part in (1, 2):
    if _part not in _parts_seqs:
        _t2.join()
        _parts_seqs.update(_bg)
    seq_dirs = _parts_seqs[_part][:2]
    for seq_dir in seq_dirs:
        frames = sorted((seq_dir / "img1").glob("*.jpg"))[::EVAL_STRIDE]
        if not probe_done:                 # probe on the first sequence(s)
            for img_path in frames[:PROBE_N - len(latencies_1)]:
                image = load_rgb(img_path)
                torch.cuda.synchronize(); _t0 = time.perf_counter()
                run_batch([image])
                torch.cuda.synchronize()
                latencies_1.append(time.perf_counter() - _t0)
            probe_done = len(latencies_1) >= PROBE_N
        seq_det = []
        with ThreadPoolExecutor(max_workers=4) as pool:
            loaded = pool.map(load_rgb, frames)   # prefetch overlaps GPU work
            batch = []
            def flush(batch):
                torch.cuda.synchronize(); _t0 = time.perf_counter()
                results = run_batch(batch)
                torch.cuda.synchronize()
                dt = time.perf_counter() - _t0
                counts = [len(r["scores"]) for r in results]
                scores = [float(r["scores"].mean()) for r in results if len(r["scores"])]
                return dt, counts, scores
            for image in loaded:
                batch.append(image)
                if len(batch) == BATCH:
                    dt, counts, scores = flush(batch)
                    gpu_s += dt; det_counts += counts; seq_det += counts
                    score_sums += scores; n_frames += len(batch)
                    batch = []
            if batch:
                dt, counts, scores = flush(batch)
                gpu_s += dt; det_counts += counts; seq_det += counts
                score_sums += scores; n_frames += len(batch)
        per_seq[seq_dir.name] = {
            "part": _part, "frames": len(frames),
            "detections_per_frame_mean": sum(seq_det) / max(1, len(seq_det)),
        }
        print(f"{seq_dir.name} (part {_part}): {len(frames)} frames, "
              f"{per_seq[seq_dir.name]['detections_per_frame_mean']:.1f} det/frame, "
              f"cum. throughput {n_frames / gpu_s:.1f} fps", flush=True)
    shutil.rmtree(part_dir(_part))         # done with this half — free the disk
WALL_S = time.time() - t_bench
print(f"\nbenchmarked {n_frames} frames across both parts in {WALL_S / 60:.1f} min "
      f"(GPU-timed throughput {n_frames / gpu_s:.1f} fps)")

In [ ]:
# --- 7. Verdict ---------------------------------------------------------------
import statistics
assert n_frames > 0, "no frames benchmarked"
fps = n_frames / gpu_s
lat1 = 1000 * statistics.mean(latencies_1)
assert fps > 2, f"implausible batched throughput {fps:.1f} fps"
assert lat1 < 1000, f"implausible single-image latency {lat1:.0f} ms"
assert 1 <= statistics.mean(det_counts) <= 30, \
    f"implausible detections/frame {statistics.mean(det_counts):.1f} — model or mount broken?"
print("=" * 62)
print(f"SMOKETEST PASSED — both parts mounted; {n_frames} frames, "
      f"batched fp16 throughput {fps:.1f} fps, single-image {lat1:.0f} ms; "
      f"{statistics.mean(det_counts):.1f} det/frame; "
      f"HF_TOKEN {'present' if os.environ.get('HF_TOKEN') else 'MISSING (card update would be skipped)'}. "
      "Environment ready for the full testing/index run.")
print("=" * 62)